In [1]:
!pip -q install openai gradio sentence-transformers faiss-cpu pymupdf python-docx pandas openpyxl

In [41]:

import os, re, json, hashlib, shutil, email
from pathlib import Path
from datetime import datetime, date
from typing import List, Dict, Any

from google.colab import userdata

# Load API key from Google Colab Secrets
API_KEY = userdata.get("API_KEY")

# Navigate Labs API configuration
BASE_URL = "https://nexusapi.navigatelabs.ai/v1"
CHAT_MODEL = "nova-micro"

# Embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Project storage
DATA_DIR = Path("/content/shopmemory_data")
UPLOAD_DIR = DATA_DIR / "uploads"
INDEX_DIR = DATA_DIR / "index"

DATA_DIR.mkdir(exist_ok=True)
UPLOAD_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 8
OUTDATED_AFTER_DAYS = 90

if API_KEY:
    print("API KEY CONNECTED SUCCESSFULLY!")
else:
    raise ValueError("API_KEY not found in Colab Secrets!")


API KEY CONNECTED SUCCESSFULLY!


In [42]:

from google.colab import userdata

API_KEY = userdata.get("API_KEY")

BASE_URL = "https://nexusapi.navigatelabs.ai/v1"

CHAT_MODEL = "nova-micro"

if API_KEY:
    print("API Key Connected Successfully!")
else:
    print("API Key Not Found!")


API Key Connected Successfully!


In [43]:

from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("API_KEY"),
    base_url="https://nexusapi.navigatelabs.ai/v1"
)

models = client.models.list()

for model in models.data:
    print(model.id)


nano-banana
whisper-1
nova-micro
text-embedding-3-small
gpt-4o-mini-tts
gpt-4.1-nano


In [44]:

from openai import OpenAI

def get_client():

    if not API_KEY:
        raise ValueError("API Key is missing!")

    return OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL
    )


def llm_chat(system_prompt, user_prompt, temperature=0.1):

    client = get_client()

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    return response.choices[0].message.content or ""


In [45]:
result = llm_chat(
    "You are ShopMemory AI.",
    "Introduce yourself in one sentence."
)

print("API CONNECTION SUCCESSFUL!")
print(result)

API CONNECTION SUCCESSFUL!
I'm ShopMemory AI, designed to help you effortlessly manage and retrieve your shopping lists and memories.


In [46]:

import fitz
import pandas as pd
from docx import Document as DocxDocument

SUPPORTED = {".pdf", ".docx", ".xlsx", ".csv", ".txt", ".eml"}

def extract_text(file_path: str) -> str:
    path = Path(file_path)
    ext = path.suffix.lower()

    if ext == ".pdf":
        doc = fitz.open(path)
        pages = []
        for i, page in enumerate(doc):
            pages.append(f"[Page {i+1}]\n{page.get_text('text')}")
        return "\n".join(pages)

    if ext == ".docx":
        doc = DocxDocument(path)
        parts = [p.text for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                parts.append(" | ".join(cell.text for cell in row.cells))
        return "\n".join(parts)

    if ext == ".xlsx":
        sheets = pd.read_excel(path, sheet_name=None, dtype=str)
        return "\n\n".join(
            f"[Sheet: {name}]\n{df.fillna('').to_csv(index=False)}"
            for name, df in sheets.items()
        )

    if ext == ".csv":
        return pd.read_csv(path, dtype=str).fillna("").to_csv(index=False)

    if ext == ".txt":
        return path.read_text(encoding="utf-8", errors="ignore")

    if ext == ".eml":
        with open(path, "rb") as f:
            msg = email.message_from_binary_file(f)
        parts = [
            f"Subject: {msg.get('Subject', '')}",
            f"From: {msg.get('From', '')}",
            f"Date: {msg.get('Date', '')}"
        ]
        body = []
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    payload = part.get_payload(decode=True)
                    if payload:
                        body.append(payload.decode(errors="ignore"))
        else:
            payload = msg.get_payload(decode=True)
            if payload:
                body.append(payload.decode(errors="ignore"))
        return "\n".join(parts + body)

    raise ValueError(f"Unsupported file type: {ext}")

def split_into_chunks(text: str, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    if not text:
        return []
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        # Prefer a paragraph or sentence boundary when practical.
        if end < len(text):
            boundary = max(text.rfind("\n", start, end), text.rfind(". ", start, end))
            if boundary > start + size * 0.55:
                end = boundary + 1
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


In [47]:

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedder = SentenceTransformer(EMBEDDING_MODEL)
documents: List[Dict[str, Any]] = []
faiss_index = None

def save_knowledge_base():
    global faiss_index
    if not documents:
        faiss_index = None
        return
    vectors = embedder.encode(
        [d["text"] for d in documents],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")
    faiss_index = faiss.IndexFlatIP(vectors.shape[1])
    faiss_index.add(vectors)
    faiss.write_index(faiss_index, str(INDEX_DIR / "shopmemory.faiss"))
    with open(INDEX_DIR / "documents.json", "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)

def load_knowledge_base():
    global documents, faiss_index
    meta = INDEX_DIR / "documents.json"
    idx = INDEX_DIR / "shopmemory.faiss"
    if meta.exists() and idx.exists():
        with open(meta, encoding="utf-8") as f:
            documents = json.load(f)
        faiss_index = faiss.read_index(str(idx))
        print(f"Loaded {len(documents)} chunks.")
    else:
        print("No saved knowledge base found. Upload documents to begin.")

def add_document(file_path: str):
    path = Path(file_path)
    if path.suffix.lower() not in SUPPORTED:
        raise ValueError("Supported formats: PDF, DOCX, XLSX, CSV, TXT, EML")
    raw = extract_text(str(path))
    chunks = split_into_chunks(raw)
    if not chunks:
        raise ValueError(f"No readable text found in {path.name}. Scanned PDFs may need OCR.")
    # Replace existing chunks from a file with the same filename.
    global documents
    documents = [d for d in documents if d["source"] != path.name]
    file_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    for i, chunk in enumerate(chunks, 1):
        documents.append({
            "text": chunk,
            "source": path.name,
            "chunk": i,
            "file_hash": file_hash,
            "uploaded_at": datetime.now().isoformat(timespec="seconds")
        })
    save_knowledge_base()
    return len(chunks)

load_knowledge_base()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 1 chunks.


In [48]:

def retrieve(query: str, top_k=TOP_K):
    if faiss_index is None or not documents:
        return []
    qv = embedder.encode([query], convert_to_numpy=True,
                         normalize_embeddings=True).astype("float32")
    k = min(top_k, len(documents))
    scores, ids = faiss_index.search(qv, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        item = dict(documents[int(idx)])
        item["score"] = float(score)
        results.append(item)
    return results

def format_evidence(results):
    blocks = []
    for i, item in enumerate(results, 1):
        blocks.append(
            f"[S{i}] Source: {item['source']} | Chunk: {item['chunk']} | "
            f"Similarity: {item['score']:.3f}\n{item['text']}"
        )
    return "\n\n".join(blocks)


In [49]:

QA_SYSTEM = """You are ShopMemory AI, an assistant for small-shop business documents.
Answer ONLY from the supplied evidence. Never invent prices, dates, supplier names,
quantities, or order conditions. If evidence is insufficient, say so clearly.
When answering, cite evidence using [S1], [S2], etc. Distinguish document facts
from your own interpretation. If sources disagree, explicitly flag the conflict
and show each value with its source and date if available. Do not silently choose
one value. Mention that a quotation may be outdated only when its date is visible
and the configured age threshold is exceeded. You are not connected to live stock
or order systems unless the evidence explicitly contains updated records."""

def ask_shopmemory(question: str, top_k=TOP_K):
    question = (question or "").strip()
    if not question:
        return "Please enter a question.", []
    results = retrieve(question, top_k=top_k)
    if not results:
        return "No documents are indexed yet. Upload invoices or supplier documents first.", []
    prompt = f"""USER QUESTION:
{question}

RETRIEVED DOCUMENT EVIDENCE:
{format_evidence(results)}

Write a clear answer with evidence citations. For prices, preserve currency and units.
If multiple prices appear, compare them and identify any disagreement."""
    answer = llm_chat(QA_SYSTEM, prompt)
    return answer, results


In [50]:

PRICE_SYSTEM = """You analyze supplier quotations and invoices.
Use ONLY the evidence provided. Extract comparable item/price information.
Do not assume two differently named products are identical. Compare only when
item descriptions and units are sufficiently similar; otherwise label them
'needs manual verification'. Preserve currency, unit, supplier and document date.
Identify conflicting prices, different units, and potentially outdated quotations.
A quote is potentially outdated if its document date is more than the configured
number of days old. Missing dates must be marked 'date not found'.
Return readable Markdown, not unsupported guesses. Cite every claim with [S#]."""

def compare_supplier_prices(item_or_question: str, top_k=12):
    results = retrieve(item_or_question, top_k=top_k)
    if not results:
        return "No matching evidence found. Upload supplier quotations first.", []
    prompt = f"""Compare supplier prices relevant to: {item_or_question}

Outdated quotation threshold: {OUTDATED_AFTER_DAYS} days from today's date ({date.today()}).

Evidence:
{format_evidence(results)}

Output:
1. A comparison table with item, supplier, quoted price, unit, document date, source.
2. Conflicting prices (only where comparison is valid).
3. Potentially outdated quotations.
4. Missing information / manual checks needed.
5. A short conclusion. Cite every row or statement with [S#]."""
    return llm_chat(PRICE_SYSTEM, prompt), results


In [51]:

def document_summary():
    if not documents:
        return "No documents uploaded yet."
    unique = {}
    for d in documents:
        unique[d["source"]] = d
    rows = ["| Document | Chunks | Last indexed |", "|---|---:|---|"]
    for name in sorted({d["source"] for d in documents}):
        items = [d for d in documents if d["source"] == name]
        rows.append(f"| {name} | {len(items)} | {items[-1]['uploaded_at']} |")
    return f"**{len(unique)} documents · {len(documents)} searchable chunks**\n\n" + "\n".join(rows)

def source_details(results):
    if not results:
        return "No evidence retrieved."
    lines = []
    for i, d in enumerate(results, 1):
        preview = d["text"][:650].replace("\n", " ")
        lines.append(
            f"**[S{i}] {d['source']} — chunk {d['chunk']}** "
            f"(similarity {d['score']:.3f})\n\n{preview}"
        )
    return "\n\n---\n\n".join(lines)

def upload_and_index(files):
    if not files:
        return "Choose one or more files first.", document_summary()
    messages = []
    for f in files:
        src = Path(f)
        dest = UPLOAD_DIR / src.name
        shutil.copy2(src, dest)
        try:
            count = add_document(str(dest))
            messages.append(f"✅ {src.name}: indexed {count} chunks")
        except Exception as e:
            messages.append(f"❌ {src.name}: {e}")
    return "\n".join(messages), document_summary()

def chat_action(question):
    answer, evidence = ask_shopmemory(question)
    return answer, source_details(evidence)

def compare_action(query):
    answer, evidence = compare_supplier_prices(query)
    return answer, source_details(evidence)

def clear_chat():
    return "", ""


In [ ]:

import gradio as gr

CUSTOM_CSS = """
.gradio-container {max-width: 1250px !important; margin: auto !important;}
#hero {padding: 18px; border-radius: 14px; background: linear-gradient(110deg,#102a43,#176b67); color: white;}
#hero h1 {margin-bottom: 5px;}
"""

with gr.Blocks(title="ShopMemory AI", theme=gr.themes.Soft(), css=CUSTOM_CSS) as demo:
    gr.HTML("""
    <div id="hero">
      <h1>🧾 ShopMemory AI</h1>
      <p>Find answers hidden in bills, supplier quotations and business messages.</p>
      <small>Evidence-backed business document intelligence</small>
    </div>
    """)

    with gr.Tab("📊 Overview"):
        overview = gr.Markdown(document_summary())
        refresh = gr.Button("Refresh dashboard")
        refresh.click(fn=document_summary, outputs=overview)
        gr.Markdown("""
        **How it works:** Upload business documents → ask questions → inspect source evidence.

        **Note:** This dashboard searches uploaded records. It does not represent live inventory.
        """)

    with gr.Tab("📁 Document Library"):
        files_input = gr.File(
            label="Upload invoices, quotations, bills or messages",
            file_count="multiple",
            type="filepath",
            file_types=[".pdf", ".docx", ".xlsx", ".csv", ".txt", ".eml"]
        )
        upload_btn = gr.Button("Upload and index documents", variant="primary")
        upload_status = gr.Markdown()
        library_status = gr.Markdown(document_summary())
        upload_btn.click(
            fn=upload_and_index,
            inputs=files_input,
            outputs=[upload_status, library_status]
        )

    with gr.Tab("💬 Ask ShopMemory"):
        question = gr.Textbox(
            label="Ask a question",
            placeholder="Example: What price did Supplier A quote for rice last month?",
            lines=2
        )
        ask_btn = gr.Button("Find answer", variant="primary")
        answer = gr.Markdown()
        evidence = gr.Markdown(label="Retrieved evidence")
        ask_btn.click(fn=chat_action, inputs=question, outputs=[answer, evidence])
        question.submit(fn=chat_action, inputs=question, outputs=[answer, evidence])

    with gr.Tab("⚖️ Price Comparison"):
        compare_query = gr.Textbox(
            label="Product or comparison request",
            placeholder="Example: Compare the quoted prices for 1 kg basmati rice",
            lines=2
        )
        compare_btn = gr.Button("Compare supplier prices", variant="primary")
        compare_result = gr.Markdown()
        compare_evidence = gr.Markdown()
        compare_btn.click(
            fn=compare_action,
            inputs=compare_query,
            outputs=[compare_result, compare_evidence]
        )

    with gr.Tab("🔎 Evidence Explorer"):
        evidence_query = gr.Textbox(
            label="Search across uploaded documents",
            placeholder="Enter a supplier, product, price or order condition"
        )
        evidence_btn = gr.Button("Retrieve evidence")
        evidence_output = gr.Markdown()
        def evidence_action(q):
            found = retrieve(q, top_k=12)
            return source_details(found)
        evidence_btn.click(fn=evidence_action, inputs=evidence_query, outputs=evidence_output)

    gr.Markdown(
        "ShopMemory AI · RAG-powered document assistant · "
        "Always verify important financial decisions against the original document."
    )

demo.launch(share=True, debug=True)


/tmp/ipykernel_722/2396839270.py:9: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="ShopMemory AI", theme=gr.themes.Soft(), css=CUSTOM_CSS) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8344434994c0820f26.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
